In [1]:
import matplotlib.pyplot as plt

In [2]:
import sys
import os

sys.path.append("./modules/")

from BottleCropper import BottleCropper
from JinaBottleVectorizer import JinaBottleVectorizer
from ImageVectorDB import ImageVectorDB

In [3]:
cropper = BottleCropper()
vectorizer = JinaBottleVectorizer()

INFO:BottleCropper:Загружаем модель: yolo26s.pt


Загрузка модели jinaai/jina-embeddings-v5-omni-small...


INFO:httpx:HTTP Request: HEAD https://huggingface.co/jinaai/jina-embeddings-v5-omni-small/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/jinaai/jina-embeddings-v5-omni-small/5c54692f22e186fc12ea38a9193e3ff9e1cda2a3/config.json?%2Fjinaai%2Fjina-embeddings-v5-omni-small%2Fresolve%2Fmain%2Fconfig.json=&etag=%22d0d2616a9ac26550e34f5e62291ca7924881fb2b%22 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/jinaai/jina-embeddings-v5-omni-small/resolve/main/modeling_jina_embeddings_v5_omni.py "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/jinaai/jina-embeddings-v5-omni-small/5c54692f22e186fc12ea38a9193e3ff9e1cda2a3/modeling_jina_embeddings_v5_omni.py?%2Fjinaai%2Fjina-embeddings-v5-omni-small%2Fresolve%2Fmain%2Fmodeling_jina_embeddings_v5_omni.py=&etag=%2223d5269ba6734d92fb7ccbcf47a8eeb616b3c43d%22 "HTTP/1.1 200 OK"
INFO:h

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/629 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/jinaai/jina-embeddings-v5-omni-small/resolve/main/processor_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/jinaai/jina-embeddings-v5-omni-small/5c54692f22e186fc12ea38a9193e3ff9e1cda2a3/processor_config.json?%2Fjinaai%2Fjina-embeddings-v5-omni-small%2Fresolve%2Fmain%2Fprocessor_config.json=&etag=%22c2a373f81d9ade85e6aa5ddc1d25e590d0bae606%22 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/jinaai/jina-embeddings-v5-omni-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/jinaai/jina-embeddings-v5-omni-small/resolve/main/processor_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/jinaai/jina-embeddings-v5-omni-small/5c54692f22e186fc12ea38a9193e3ff9e1cda2a3/proc

Модель успешно загружена!


In [4]:
db = ImageVectorDB()

INFO:httpx:HTTP Request: GET http://localhost:6333/collections "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/images_embeddings "HTTP/1.1 200 OK"


Коллекция 'images_embeddings' успешно создана.


In [5]:
images = list(filter(lambda x: x.endswith('.jpg'), os.listdir('./dataset/train/images/')))

In [6]:
failed = []

for image in images:
    image_file = os.path.join("./dataset/train/images/", image)
    bottle = cropper.crop_image(image_file)
    if not bottle:
        failed.append(image_file)
        continue
    embedding = vectorizer.vectorize_array(bottle[0])
    db.upsert_images(image_names=[image_file], vectors=[embedding])


INFO:httpx:HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
C:\Users\drand\anaconda3\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.17.1. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(
INFO:BottleCropper:Бутылка не найдена на изображении -7_jpg.rf.s5HtodavB0bCc1qN66pA.jpg
INFO:BottleCropper:Бутылка не найдена на изображении 0688f063-793d-4b01-a46e-05d8ea6d0e4b_jpg.rf.VkoCkkC7bPh5uxIUYp3M.jpg
INFO:BottleCropper:Вырезана самая большая бутылка: 'bottle', уверенность 40.45%, площадь 73571 px²
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/images_embeddings/points?wait=true "HTTP/1.1 200 OK"
INFO:BottleCropper:Вырезана самая большая бутылка: 'bottle', уверенность 36.99%, площадь 12406 px²
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/images_embeddings/points?wait=true

In [15]:
from IPython.display import clear_output, Image, display

In [33]:
from io import BytesIO

In [85]:
model = cropper.model

for fail in failed:
    r = model(fail)
    r[0].save(f'./failed/fail_{fail.split('/')[-1]}.png')

In [86]:
for fail in failed:
    clear_output()
    display(Image(f'./failed/fail_{fail.split('/')[-1]}.png'))
    input("pause")

FileNotFoundError: No such file or directory: './failed/fail_./dataset/train/images/-7_jpg.rf.s5HtodavB0bCc1qN66pA.jpg.png'

FileNotFoundError: No such file or directory: './failed/fail_./dataset/train/images/-7_jpg.rf.s5HtodavB0bCc1qN66pA.jpg.png'

<IPython.core.display.Image object>

KeyboardInterrupt: Interrupted by user